# NeuroGolf 2026 Solver Diagnostics

This notebook runs lightweight rule diagnostics before committing to ONNX model families. It estimates which tasks are compatible with simple transformations such as identity, color maps, cropping, mirroring, rotation, and tiling-like expansion.

# 1. Setup and Data Loading

## 1.1 Imports and Display Defaults

The diagnostics are intentionally dependency-light so they can run alongside the EDA notebook on Kaggle without extra setup.

In [1]:
from __future__ import annotations

import json
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)

PLOT_CMAP = "viridis"
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def insight(title: str, bullets: list[str]) -> None:
    clean_bullets = [b for b in bullets if b]
    if clean_bullets:
        display(Markdown("### " + title + "
" + "
".join(f"- {b}" for b in clean_bullets)))


SyntaxError: unterminated string literal (detected at line 29) (4205680074.py, line 29)

## 1.2 Data Discovery

This loader mirrors the earlier notebooks: search `/kaggle/input`, support both one-file-per-task and combined JSON layouts, and normalize tasks into a dictionary keyed by task id.

In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")
LOCAL_CANDIDATES = [Path("../input"), Path("input"), Path("data"), Path("../data")]
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")


def candidate_roots() -> list[Path]:
    roots: list[Path] = []
    if KAGGLE_INPUT.exists():
        roots.extend([p for p in KAGGLE_INPUT.iterdir() if p.is_dir()])
        roots.append(KAGGLE_INPUT)
    roots.extend([p for p in LOCAL_CANDIDATES if p.exists()])
    return roots


def find_json_files() -> list[Path]:
    files: list[Path] = []
    for root in candidate_roots():
        files.extend(root.rglob("*.json"))
    return sorted(set(files))


def is_task_payload(obj: Any) -> bool:
    return isinstance(obj, dict) and "train" in obj and "test" in obj


def normalize_task_id(path: Path, key: str | None = None) -> str:
    if key:
        key = str(key)
        return key[:-5] if key.endswith(".json") else key
    return path.stem


def load_tasks(files: list[Path]) -> dict[str, dict[str, Any]]:
    tasks: dict[str, dict[str, Any]] = {}
    for path in files:
        try:
            with path.open("r") as f:
                obj = json.load(f)
        except Exception as exc:
            print(f"Skipping {path}: {exc}")
            continue

        if is_task_payload(obj):
            tasks[normalize_task_id(path)] = obj
        elif isinstance(obj, dict):
            for key, value in obj.items():
                if is_task_payload(value):
                    tasks[normalize_task_id(path, key)] = value
        elif isinstance(obj, list):
            for idx, value in enumerate(obj, start=1):
                if is_task_payload(value):
                    tasks[f"task{idx:03d}"] = value
    return dict(sorted(tasks.items()))


json_files = find_json_files()
tasks = load_tasks(json_files)
print(f"Found {len(json_files):,} JSON files")
print(f"Loaded {len(tasks):,} tasks")

insight("Loading Insights", [
    f"Loaded {len(tasks):,} normalized tasks for solver diagnostics.",
    "If no tasks are loaded, attach the NeuroGolf competition data before running the diagnostics.",
])


# 2. Diagnostic Feature Table

## 2.1 Task-Level Structure

This table captures the structural features needed for solver triage: train/test counts, shape changes, palette changes, area ratios, and background-token prevalence.

In [ ]:
def arr(grid: list[list[int]]) -> np.ndarray:
    return np.asarray(grid, dtype=np.int64)


def grid_shape(grid: list[list[int]]) -> tuple[int, int]:
    a = arr(grid)
    return tuple(a.shape) if a.ndim == 2 else (0, 0)


def grid_colors(grid: list[list[int]]) -> set[int]:
    a = arr(grid)
    return set(map(int, np.unique(a))) if a.size else set()


def background_share(grid: list[list[int]], token: int = 0) -> float:
    a = arr(grid)
    return float((a == token).mean()) if a.size else np.nan


def task_features(task_id: str, task: dict[str, Any]) -> dict[str, Any]:
    train = task.get("train", [])
    test = task.get("test", [])
    input_shapes = [grid_shape(pair["input"]) for pair in train + test if "input" in pair]
    output_shapes = [grid_shape(pair["output"]) for pair in train + test if "output" in pair]
    train_shape_changes = [grid_shape(pair["input"]) != grid_shape(pair["output"]) for pair in train if "input" in pair and "output" in pair]
    input_colors = set().union(*(grid_colors(pair["input"]) for pair in train + test if "input" in pair)) if train or test else set()
    output_colors = set().union(*(grid_colors(pair["output"]) for pair in train + test if "output" in pair)) if output_shapes else set()
    max_input_area = max((r * c for r, c in input_shapes), default=0)
    max_output_area = max((r * c for r, c in output_shapes), default=0)
    train_bg = [background_share(pair["input"]) for pair in train if "input" in pair]

    return {
        "task_id": task_id,
        "n_train": len(train),
        "n_test": len(test),
        "has_test_outputs": bool(test) and all("output" in pair for pair in test),
        "shape_changes_in_train": any(train_shape_changes),
        "input_shapes": sorted(set(input_shapes)),
        "output_shapes": sorted(set(output_shapes)),
        "max_input_area": max_input_area,
        "max_output_area": max_output_area,
        "area_ratio": max_output_area / max_input_area if max_input_area else np.nan,
        "n_input_colors": len(input_colors),
        "n_output_colors": len(output_colors),
        "input_colors": tuple(sorted(input_colors)),
        "output_colors": tuple(sorted(output_colors)),
        "introduced_colors": tuple(sorted(output_colors - input_colors)),
        "removed_colors": tuple(sorted(input_colors - output_colors)),
        "mean_input_background_share": float(np.nanmean(train_bg)) if train_bg else np.nan,
    }


features_df = pd.DataFrame([task_features(task_id, task) for task_id, task in tasks.items()])
if not features_df.empty:
    features_df["area_group"] = pd.cut(
        features_df["area_ratio"],
        bins=[-np.inf, 0.5, 0.95, 1.05, 2.0, np.inf],
        labels=["strong compression", "mild compression", "same area", "mild expansion", "strong expansion"],
    )
    features_df["palette_relation"] = np.select(
        [
            features_df["introduced_colors"].map(len).eq(0) & features_df["removed_colors"].map(len).eq(0),
            features_df["introduced_colors"].map(len).gt(0) & features_df["removed_colors"].map(len).eq(0),
            features_df["introduced_colors"].map(len).eq(0) & features_df["removed_colors"].map(len).gt(0),
        ],
        ["same palette", "introduces colors", "removes colors"],
        default="introduces and removes colors",
    )

display(features_df.head(10))
insight("Feature Table Insights", [
    f"Built diagnostic features for {len(features_df):,} tasks.",
    "Use this table to prioritize solver families by structure rather than by task id order.",
])


# 3. Simple Solver Hypotheses

## 3.1 Same-Shape Solvers

These checks ask whether every training pair in a task is compatible with very simple same-shape transformations. They are intentionally strict: a solver only passes when it explains all training examples for the task.

In [ ]:
def equal_grid(a: np.ndarray, b: np.ndarray) -> bool:
    return a.shape == b.shape and np.array_equal(a, b)


def infer_global_color_map(pairs: list[dict[str, Any]]) -> dict[int, int] | None:
    mapping: dict[int, int] = {}
    for pair in pairs:
        x = arr(pair["input"])
        y = arr(pair["output"])
        if x.shape != y.shape:
            return None
        for src, dst in zip(x.ravel(), y.ravel()):
            src_i = int(src)
            dst_i = int(dst)
            if src_i in mapping and mapping[src_i] != dst_i:
                return None
            mapping[src_i] = dst_i
    return mapping


def apply_color_map(x: np.ndarray, mapping: dict[int, int]) -> np.ndarray:
    y = x.copy()
    for src, dst in mapping.items():
        y[x == src] = dst
    return y


def solver_identity(train_pairs: list[dict[str, Any]]) -> bool:
    return bool(train_pairs) and all(equal_grid(arr(pair["input"]), arr(pair["output"])) for pair in train_pairs)


def solver_global_color_map(train_pairs: list[dict[str, Any]]) -> bool:
    mapping = infer_global_color_map(train_pairs)
    if mapping is None:
        return False
    return all(equal_grid(apply_color_map(arr(pair["input"]), mapping), arr(pair["output"])) for pair in train_pairs)


def solver_background_to_single_color(train_pairs: list[dict[str, Any]], background: int = 0) -> bool:
    if not train_pairs:
        return False
    fill_color: int | None = None
    for pair in train_pairs:
        x = arr(pair["input"])
        y = arr(pair["output"])
        if x.shape != y.shape:
            return False
        changed = x != y
        if not np.any(changed):
            continue
        if not np.all(x[changed] == background):
            return False
        output_colors = set(map(int, np.unique(y[changed])))
        if len(output_colors) != 1:
            return False
        current = next(iter(output_colors))
        if fill_color is None:
            fill_color = current
        elif fill_color != current:
            return False
    return fill_color is not None


def solver_mirror_or_rotation(train_pairs: list[dict[str, Any]], transform: Callable[[np.ndarray], np.ndarray]) -> bool:
    return bool(train_pairs) and all(equal_grid(transform(arr(pair["input"])), arr(pair["output"])) for pair in train_pairs)


same_shape_solvers: dict[str, Callable[[list[dict[str, Any]]], bool]] = {
    "identity": solver_identity,
    "global_color_map": solver_global_color_map,
    "background_to_single_color": solver_background_to_single_color,
    "flip_horizontal": lambda pairs: solver_mirror_or_rotation(pairs, np.fliplr),
    "flip_vertical": lambda pairs: solver_mirror_or_rotation(pairs, np.flipud),
    "rotate_90": lambda pairs: solver_mirror_or_rotation(pairs, lambda x: np.rot90(x, 1)),
    "rotate_180": lambda pairs: solver_mirror_or_rotation(pairs, lambda x: np.rot90(x, 2)),
    "rotate_270": lambda pairs: solver_mirror_or_rotation(pairs, lambda x: np.rot90(x, 3)),
    "transpose": lambda pairs: solver_mirror_or_rotation(pairs, np.transpose),
}

same_shape_rows: list[dict[str, Any]] = []
for task_id, task in tasks.items():
    train_pairs = task.get("train", [])
    row = {"task_id": task_id}
    for solver_name, solver_fn in same_shape_solvers.items():
        row[solver_name] = bool(solver_fn(train_pairs))
    same_shape_rows.append(row)

same_shape_df = pd.DataFrame(same_shape_rows)
if not same_shape_df.empty:
    solver_cols = [c for c in same_shape_df.columns if c != "task_id"]
    same_shape_df["any_same_shape_solver"] = same_shape_df[solver_cols].any(axis=1)
    same_shape_coverage = same_shape_df[solver_cols + ["any_same_shape_solver"]].sum().sort_values(ascending=False).rename_axis("solver").reset_index(name="task_count")
    display(same_shape_coverage)
    display(same_shape_df.query("any_same_shape_solver == True").head(20))

    ax = same_shape_coverage.set_index("solver")["task_count"].plot(kind="bar", figsize=(10, 4), color=plt.get_cmap(PLOT_CMAP)(0.7))
    ax.set_title("Strict same-shape solver compatibility")
    ax.set_xlabel("solver hypothesis")
    ax.set_ylabel("task count")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()

    same_shape_bullets = [
        f"At least one strict same-shape solver explains {int(same_shape_df['any_same_shape_solver'].sum()):,} tasks on training pairs.",
        "Global color maps are a useful first baseline when they explain all train pairs, but they still need validation against test input structure.",
        "Low counts here are not bad; they tell us how quickly we need object-level and search-based solvers.",
    ]
else:
    same_shape_bullets = ["Same-shape solver diagnostics require loaded tasks."]
insight("Same-Shape Solver Insights", same_shape_bullets)


## 3.2 Shape-Changing Solvers

These checks look for simple output construction patterns: crop to non-background bounding box, integer scaling, and repeating/tile-like expansions. They are diagnostic labels, not final solvers yet.

In [ ]:
def crop_to_non_background(x: np.ndarray, background: int = 0) -> np.ndarray | None:
    coords = np.argwhere(x != background)
    if coords.size == 0:
        return None
    r0, c0 = coords.min(axis=0)
    r1, c1 = coords.max(axis=0) + 1
    return x[r0:r1, c0:c1]


def solver_crop_non_background(train_pairs: list[dict[str, Any]], background: int = 0) -> bool:
    if not train_pairs:
        return False
    for pair in train_pairs:
        cropped = crop_to_non_background(arr(pair["input"]), background=background)
        if cropped is None or not equal_grid(cropped, arr(pair["output"])):
            return False
    return True


def integer_scale_factor(input_shape: tuple[int, int], output_shape: tuple[int, int]) -> tuple[int, int] | None:
    ir, ic = input_shape
    or_, oc = output_shape
    if ir == 0 or ic == 0 or or_ % ir != 0 or oc % ic != 0:
        return None
    return (or_ // ir, oc // ic)


def solver_nearest_integer_scale(train_pairs: list[dict[str, Any]]) -> bool:
    if not train_pairs:
        return False
    scale: tuple[int, int] | None = None
    for pair in train_pairs:
        x = arr(pair["input"])
        y = arr(pair["output"])
        current = integer_scale_factor(tuple(x.shape), tuple(y.shape))
        if current is None or current == (1, 1):
            return False
        if scale is None:
            scale = current
        elif scale != current:
            return False
        scaled = np.repeat(np.repeat(x, current[0], axis=0), current[1], axis=1)
        if not equal_grid(scaled, y):
            return False
    return True


def solver_periodic_tile_from_input(train_pairs: list[dict[str, Any]]) -> bool:
    if not train_pairs:
        return False
    for pair in train_pairs:
        x = arr(pair["input"])
        y = arr(pair["output"])
        if y.shape[0] < x.shape[0] or y.shape[1] < x.shape[1] or x.size == 0:
            return False
        tiled = np.tile(x, (int(np.ceil(y.shape[0] / x.shape[0])), int(np.ceil(y.shape[1] / x.shape[1]))))[: y.shape[0], : y.shape[1]]
        if not equal_grid(tiled, y):
            return False
    return True


shape_solver_fns: dict[str, Callable[[list[dict[str, Any]]], bool]] = {
    "crop_non_background": solver_crop_non_background,
    "nearest_integer_scale": solver_nearest_integer_scale,
    "periodic_tile_from_input": solver_periodic_tile_from_input,
}

shape_rows: list[dict[str, Any]] = []
for task_id, task in tasks.items():
    train_pairs = task.get("train", [])
    row = {"task_id": task_id}
    for solver_name, solver_fn in shape_solver_fns.items():
        row[solver_name] = bool(solver_fn(train_pairs))
    shape_rows.append(row)

shape_solver_df = pd.DataFrame(shape_rows)
if not shape_solver_df.empty:
    shape_cols = [c for c in shape_solver_df.columns if c != "task_id"]
    shape_solver_df["any_shape_solver"] = shape_solver_df[shape_cols].any(axis=1)
    shape_coverage = shape_solver_df[shape_cols + ["any_shape_solver"]].sum().sort_values(ascending=False).rename_axis("solver").reset_index(name="task_count")
    display(shape_coverage)
    display(shape_solver_df.query("any_shape_solver == True").head(20))

    ax = shape_coverage.set_index("solver")["task_count"].plot(kind="bar", figsize=(8, 4), color=plt.get_cmap(PLOT_CMAP)(0.72))
    ax.set_title("Strict shape-changing solver compatibility")
    ax.set_xlabel("solver hypothesis")
    ax.set_ylabel("task count")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()

    shape_bullets = [
        f"At least one strict shape-changing heuristic explains {int(shape_solver_df['any_shape_solver'].sum()):,} tasks on training pairs.",
        "Crop, scale, and tile detections are good candidates for early rule solvers because they are cheap and interpretable.",
        "Tasks not covered here should move into object-level diagnostics rather than more global array transforms.",
    ]
else:
    shape_bullets = ["Shape-changing diagnostics require loaded tasks."]
insight("Shape-Changing Solver Insights", shape_bullets)


# 4. Object-Level Diagnostics

## 4.1 Connected Components by Color

This section estimates object complexity using connected components over non-background cells. It helps separate sparse object tasks from dense texture or pattern tasks.

In [ ]:
def connected_components(grid: np.ndarray, background: int = 0) -> list[dict[str, Any]]:
    visited = np.zeros(grid.shape, dtype=bool)
    components: list[dict[str, Any]] = []
    rows, cols = grid.shape
    for r in range(rows):
        for c in range(cols):
            if visited[r, c] or int(grid[r, c]) == background:
                continue
            color = int(grid[r, c])
            stack = [(r, c)]
            visited[r, c] = True
            coords = []
            while stack:
                cr, cc = stack.pop()
                coords.append((cr, cc))
                for nr, nc in [(cr - 1, cc), (cr + 1, cc), (cr, cc - 1), (cr, cc + 1)]:
                    if 0 <= nr < rows and 0 <= nc < cols and not visited[nr, nc] and int(grid[nr, nc]) == color:
                        visited[nr, nc] = True
                        stack.append((nr, nc))
            coord_arr = np.asarray(coords)
            r0, c0 = coord_arr.min(axis=0)
            r1, c1 = coord_arr.max(axis=0) + 1
            components.append({
                "color": color,
                "area": len(coords),
                "bbox_height": int(r1 - r0),
                "bbox_width": int(c1 - c0),
                "bbox_area": int((r1 - r0) * (c1 - c0)),
            })
    return components


component_rows: list[dict[str, Any]] = []
for task_id, task in tasks.items():
    for split in ["train", "test"]:
        for idx, pair in enumerate(task.get(split, []), start=1):
            if "input" not in pair:
                continue
            comps = connected_components(arr(pair["input"]), background=0)
            areas = [comp["area"] for comp in comps]
            component_rows.append({
                "task_id": task_id,
                "split": split,
                "pair_idx": idx,
                "n_components": len(comps),
                "max_component_area": max(areas, default=0),
                "mean_component_area": float(np.mean(areas)) if areas else 0.0,
                "n_component_colors": len(set(comp["color"] for comp in comps)),
            })

component_df = pd.DataFrame(component_rows)
if not component_df.empty:
    task_component_df = component_df.groupby("task_id").agg(
        mean_components=("n_components", "mean"),
        max_components=("n_components", "max"),
        mean_component_area=("mean_component_area", "mean"),
        max_component_area=("max_component_area", "max"),
        mean_component_colors=("n_component_colors", "mean"),
    ).reset_index()
    display(task_component_df.head(10))
    display(task_component_df.sort_values("max_components", ascending=False).head(15))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    task_component_df["max_components"].hist(ax=axes[0], bins=30, color=plt.get_cmap(PLOT_CMAP)(0.62))
    axes[0].set_title("Max non-background components per task")
    axes[0].set_xlabel("component count")
    axes[0].set_ylabel("task count")
    task_component_df["max_component_area"].hist(ax=axes[1], bins=30, color=plt.get_cmap(PLOT_CMAP)(0.82))
    axes[1].set_title("Max component area per task")
    axes[1].set_xlabel("cells")
    plt.tight_layout()

    component_bullets = [
        f"Median max component count is {task_component_df['max_components'].median():.1f}, with a maximum of {task_component_df['max_components'].max():.0f}.",
        "Low-component tasks are good candidates for object extraction and movement solvers.",
        "High-component tasks likely need pattern, texture, grid-line, or global counting diagnostics.",
    ]
else:
    task_component_df = pd.DataFrame()
    component_bullets = ["Component diagnostics require loaded tasks."]
insight("Component Insights", component_bullets)


# 5. Prioritization

## 5.1 Recommended Solver Backlog

This section combines diagnostic labels into a pragmatic queue for the next modeling notebooks.

In [ ]:
if not features_df.empty:
    priority_df = features_df[["task_id", "shape_changes_in_train", "area_group", "palette_relation", "n_train", "n_test", "mean_input_background_share"]].copy()
    if not same_shape_df.empty:
        priority_df = priority_df.merge(same_shape_df[["task_id", "any_same_shape_solver"]], on="task_id", how="left")
    else:
        priority_df["any_same_shape_solver"] = False
    if not shape_solver_df.empty:
        priority_df = priority_df.merge(shape_solver_df[["task_id", "any_shape_solver"]], on="task_id", how="left")
    else:
        priority_df["any_shape_solver"] = False
    if not task_component_df.empty:
        priority_df = priority_df.merge(task_component_df[["task_id", "max_components", "max_component_area"]], on="task_id", how="left")
    else:
        priority_df["max_components"] = np.nan
        priority_df["max_component_area"] = np.nan

    priority_df["recommended_track"] = np.select(
        [
            priority_df["any_same_shape_solver"].fillna(False),
            priority_df["any_shape_solver"].fillna(False),
            priority_df["shape_changes_in_train"] & priority_df["area_group"].astype(str).str.contains("compression", na=False),
            priority_df["shape_changes_in_train"] & priority_df["area_group"].astype(str).str.contains("expansion", na=False),
            priority_df["max_components"].fillna(999).le(10),
        ],
        [
            "implement simple same-shape solver",
            "implement simple shape solver",
            "deep dive crop/extract/compress",
            "deep dive expand/tile/construct",
            "deep dive object movement/selection",
        ],
        default="deep dive pattern/counting/global logic",
    )

    track_table = priority_df["recommended_track"].value_counts().rename_axis("recommended_track").reset_index(name="task_count")
    display(track_table)
    display(priority_df.head(20))

    ax = track_table.set_index("recommended_track")["task_count"].plot(kind="bar", figsize=(11, 4), color=plt.get_cmap(PLOT_CMAP)(0.76))
    ax.set_title("Recommended next solver tracks")
    ax.set_xlabel("track")
    ax.set_ylabel("task count")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()

    priority_bullets = [
        f"The largest recommended track is `{track_table.iloc[0]['recommended_track']}` with {int(track_table.iloc[0]['task_count']):,} tasks.",
        "Use this table to choose the next baseline notebook scope rather than implementing solvers in task-id order.",
        "Solver notebooks should report train-fit coverage by track before attempting ONNX export.",
    ]
else:
    priority_df = pd.DataFrame()
    priority_bullets = ["Prioritization requires diagnostic features."]
insight("Prioritization Insights", priority_bullets)


## 5.2 Export Diagnostics

Export compact diagnostic tables to `/kaggle/working` so modeling notebooks can reuse solver labels without recomputing every chart.

In [ ]:
if not features_df.empty:
    feature_path = OUTPUT_DIR / "neurogolf_solver_diagnostic_features.csv"
    priority_path = OUTPUT_DIR / "neurogolf_solver_priority_tracks.csv"
    features_df.to_csv(feature_path, index=False)
    priority_df.to_csv(priority_path, index=False)
    print(f"Wrote {feature_path}")
    print(f"Wrote {priority_path}")
else:
    print("No diagnostics exported because no tasks were loaded.")

insight("Export Insights", [
    "Diagnostics are ready for downstream solver notebooks." if not features_df.empty else "No diagnostic artifacts were written.",
    "Use the priority-track CSV to scope baseline solver implementation and ONNX export experiments.",
])
